# Final-training calibration

Learning curves used to choose the fixed final-training epoch without accessing locked test subjects.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

dataset_labels = {'bnci2014_001': 'BNCI', 'lee2019_mi': 'Lee', 'physionet_mi': 'PhysioNet'}
model_labels = {
    'compact_eegnet': 'Compact', 'deep_separable': 'Deep', 'wide_eegnet': 'Wide',
    'shallow_temporal': 'Shallow', 'dilated_temporal': 'Dilated',
}
model_order = list(model_labels)
source_colors = {'bnci2014_001': '#2563eb', 'lee2019_mi': '#059669', 'physionet_mi': '#dc2626'}
DATA_ROOT = Path('../data') if Path('../data').exists() else Path('data')

## NAS final-training calibration

In [ ]:
DATA_ROOT = Path('../data') if Path('../data').exists() else Path('data')
calibration_files = sorted(DATA_ROOT.glob('seed-*/eeg-parzen/calibration/*/calibration_results.jsonl'))
if len(calibration_files) != 3:
    raise RuntimeError(f'Expected three calibration files, found {len(calibration_files)}')
calibration = pd.DataFrame(
    {
        'seed': entry['seed'],
        'source': entry['source_dataset'],
        'target': entry['target_dataset'],
        'epoch': entry['epoch'],
        'train': entry['train']['balanced_accuracy'],
        'validation': entry['validation']['balanced_accuracy'],
    }
    for path in calibration_files
    for entry in (json.loads(line) for line in path.open() if line.strip())
)
calibration['group'] = calibration.apply(
    lambda row: 'Diagonal' if row['source'] == row['target'] else 'Transfer', axis=1
)
overall = calibration.groupby('epoch').agg(
    train_mean=('train', 'mean'), train_sd=('train', 'std'),
    validation_mean=('validation', 'mean'), validation_sd=('validation', 'std'),
)
overall['generalization_gap'] = overall['train_mean'] - overall['validation_mean']
target_summary = calibration.groupby(['target', 'epoch'])['validation'].mean().reset_index()
component_summary = calibration.groupby(['target', 'source', 'epoch'])['validation'].mean().reset_index()
fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
mean_axis = axes[0, 0]
for metric, label, color in [('train', 'Train', 'tab:blue'), ('validation', 'Validation', 'tab:orange')]:
    mean, sd = overall[f'{metric}_mean'], overall[f'{metric}_sd']
    mean_axis.plot(overall.index, mean, marker='o', linewidth=2, label=f'{label} mean', color=color)
    mean_axis.fill_between(overall.index, mean - sd, mean + sd, alpha=0.15, color=color)
for seed, group in calibration.groupby('seed'):
    curve = group.groupby('epoch')['validation'].mean()
    mean_axis.plot(curve.index, curve, linestyle=':', alpha=0.65, label=f'Validation seed {seed}')
for epoch, value in overall['validation_mean'].items():
    mean_axis.annotate(f'{value:.3f}', (epoch, value), xytext=(0, -13), textcoords='offset points', ha='center', fontsize=8)
dataset_labels = {'bnci2014_001': 'BNCI', 'lee2019_mi': 'Lee', 'physionet_mi': 'PhysioNet'}
target_axes = dict(zip(dataset_labels, [axes[0, 1], axes[1, 0], axes[1, 1]], strict=True))
for target, axis in target_axes.items():
    target_components = component_summary[component_summary['target'] == target]
    for source, group in target_components.groupby('source'):
        axis.plot(group['epoch'], group['validation'], marker='o', alpha=0.7, label=f'{dataset_labels[source]}-selected')
    target_mean = target_summary[target_summary['target'] == target]
    axis.plot(target_mean['epoch'], target_mean['validation'], color='black', linewidth=3, label='Target mean')
    axis.set_title(f'{dataset_labels[target]} target')
for axis in axes.flat:
    axis.set(xlabel='Epoch', xticks=range(10, 101, 10))
    axis.grid(alpha=0.25)
    axis.legend()
mean_axis.set(ylabel='Balanced accuracy', title='Mean learning curve (27 models, ± sample SD)')
for axis in target_axes.values():
    axis.set_ylabel('Validation balanced accuracy')
plt.show()
target_table = target_summary.pivot(index='epoch', columns='target', values='validation').rename(columns=dataset_labels)
component_table = component_summary.pivot(index='epoch', columns=['target', 'source'], values='validation').rename(columns=dataset_labels)
# display(overall.round(4), target_table.round(4), component_table.round(4))

## Fixed-baseline final-training calibration

In [ ]:
baseline_calibration_files = sorted(DATA_ROOT.glob('seed-*/benchmarking/calibration/*/calibration_results.jsonl'))
if len(baseline_calibration_files) != 3:
    raise RuntimeError(f'Expected three baseline calibration files, found {len(baseline_calibration_files)}')
baseline_calibration = pd.DataFrame(
    {
        'seed': entry['seed'], 'dataset': entry['dataset'], 'model': entry['baseline'],
        'epoch': entry['epoch'], 'train': entry['train']['balanced_accuracy'],
        'validation': entry['validation']['balanced_accuracy'],
    }
    for path in baseline_calibration_files
    for entry in (json.loads(line) for line in path.open() if line.strip())
)
baseline_overall = baseline_calibration.groupby('epoch').agg(
    train_mean=('train', 'mean'), train_sd=('train', 'std'),
    validation_mean=('validation', 'mean'), validation_sd=('validation', 'std'),
)
baseline_components = (
    baseline_calibration.groupby(['dataset', 'model', 'epoch'])['validation'].mean().reset_index()
)
baseline_targets = baseline_calibration.groupby(['dataset', 'epoch'])['validation'].mean().reset_index()
fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
mean_axis = axes[0, 0]
for metric, label, color in [('train', 'Train', 'tab:blue'), ('validation', 'Validation', 'tab:orange')]:
    mean, sd = baseline_overall[f'{metric}_mean'], baseline_overall[f'{metric}_sd']
    mean_axis.plot(baseline_overall.index, mean, marker='o', linewidth=2, label=f'{label} mean', color=color)
    mean_axis.fill_between(baseline_overall.index, mean - sd, mean + sd, alpha=0.15, color=color)
for seed, group in baseline_calibration.groupby('seed'):
    curve = group.groupby('epoch')['validation'].mean()
    mean_axis.plot(curve.index, curve, linestyle=':', alpha=0.65, label=f'Validation seed {seed}')
for epoch, value in baseline_overall['validation_mean'].items():
    mean_axis.annotate(f'{value:.3f}', (epoch, value), xytext=(0, -13), textcoords='offset points', ha='center', fontsize=8)
baseline_axes = dict(zip(dataset_labels, [axes[0, 1], axes[1, 0], axes[1, 1]], strict=True))
for dataset, axis in baseline_axes.items():
    components = baseline_components[baseline_components['dataset'] == dataset]
    for model, group in components.groupby('model'):
        axis.plot(group['epoch'], group['validation'], marker='o', alpha=0.7, label=model_labels[model])
    target = baseline_targets[baseline_targets['dataset'] == dataset]
    axis.plot(target['epoch'], target['validation'], color='black', linewidth=3, label='Dataset mean')
    axis.set_title(f'{dataset_labels[dataset]} baselines')
for axis in axes.flat:
    axis.set(xlabel='Epoch', xticks=range(10, 101, 10))
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)
mean_axis.set(ylabel='Balanced accuracy', title='Mean baseline learning curve (45 models, ± sample SD)')
for axis in baseline_axes.values():
    axis.set_ylabel('Validation balanced accuracy')
plt.show()